# Recommended beam-beam simulation configuration

In a strongly disrupted collision the two beams focus each other, and the vertical size at the
centre of the crossing falls to a fraction of the incoming one. Most of the luminosity is
produced there, so a grid chosen to resolve the incoming beam can miss the pinched core, and a
macroparticle count that looks generous can leave that core under-populated. Either one returns
a luminosity that appears converged and is not.

This notebook turns beam parameters into a configuration that satisfies the requirements
established for that regime, for WarpX and GUINEA-PIG++: the cell counts `n_x`, `n_y`, `n_z`,
the number of steps `n_t`, and the macroparticle count `n_m`. Give it the energy, bunch
population, bunch length, emittances and beta functions of your machine, and optionally the box
extents you want to run; it reports the configuration and flags every input that falls outside
the range the underlying study covered.

The requirements, the constants and the study they come from are described in the paper,
*Evaluating beam-beam simulations in the pursuit of designing next generation colliders*
(arXiv link to be added on release).

## How the recommendation is made

**Inputs**

- beam energy, particles per bunch, bunch length `sigma_z`
- normalised emittances `eps_x`, `eps_y` and the IP beta functions `beta_x*`, `beta_y*`
- optionally the box extents `(c_x, c_y, c_z)`, in units of the beam size; the defaults are the
  ones this study ran

**Calculation**

1. **Beam sizes and disruption.** `sigma* = sqrt(beta* eps/gamma)` in each plane, then
   `D_y = 2 N r_e sigma_z / (gamma sigma_y* (sigma_x* + sigma_y*))`, and `D_x` with `sigma_x*` in
   place of `sigma_y*`.

2. **Depth of the pinch.** The opposing bunch acts as a focusing channel,
   `K_y(s) = [D_y/(sqrt(3) sigma_z^2)] g(s)` with `g` the Gaussian longitudinal profile.
   Integrating `y'' + K_y(s) y = 0` from `s = -3 sigma_z`, where the incoming beam has drifted to
   `beta = beta_y* + s^2/beta_y*`, and taking the **first** local minimum of `beta(s)` gives the
   compression `R = sigma_y*/sigma_y^min`. Over the fitted range this is `R = Lambda D_y^q_p`,
   which is `Lambda = 1.86`, `q_p = 0.215` at this machine's `beta_y*/sigma_z = 1.2`. Both
   constants come out of that integration, so a different `beta_y*/sigma_z` gives different ones.

3. **Vertical cells.** `n_y^req = C_y D_y^q_n` with `q_n = q_p + 1/4` and
   `C_y = 2 c_y kappa Lambda`. Resolving the core needs `kappa` cells per core sigma; the extra
   `1/4` covers the roughly `sqrt(D_y)` passes the beam makes through the core, whose deposition
   errors add coherently.

4. **Macroparticles.** The central cell of the pinched core has to hold `P(D_y) = P_0 D_y^s`
   macroparticles. Its occupancy is `[8 c_x c_y c_z/(2 pi)^(3/2)] n_m/(n_x n_y n_z)` multiplied
   by the compression, which gives `n_m^req = C_m D_y^(s-q_p) n_x n_y n_z` with
   `C_m = (2 pi)^(3/2) P_0/(8 c_x c_y c_z Lambda)`.

5. **Box and rounding.** The box is `+-c sigma`, so the cell size is `2 c sigma/n`: a larger box
   needs proportionally more cells to keep the same resolution, and `n_x`, `n_y`, `n_z` each
   scale with their own multiplier. `n_y` is rounded up to a power of two, and `n_m` follows from
   the rounded cell counts.

**Outputs**

- the configuration to run: `n_x`, `n_y`, `n_z`, `n_t` and `n_m`
- what it was derived from: `D_y`, `D_x`, `sigma_x*`, `sigma_y*`, `beta_y*/sigma_z`, and the cut
  multipliers
- a warning for every input outside what the study covered

Both laws use the conservative locus: the fitted line raised until every tuning point sits
beneath it. The constants and their provenance are in `recommend.py`.

In [1]:
from recommend import recommend, selftest, REFERENCE, HOURGLASS_REF

selftest()

R(D_y) reproduces the reference values to 5.1e-04
both recommended-configuration tables reproduced


True

## The configurations in the paper

The reference machine, C3-250, across the tuned emittance range. This reproduces the
recommendation tables of the paper.

In [2]:
import warnings

reference_beam = {k: v for k, v in REFERENCE.items() if k != 'name'}

hdr = f"{'eps_y [nm]':>10} {'D_y':>7} | {'GP n_y':>7} {'GP n_m':>12} | {'WX n_y':>7} {'WX n_m':>10} | notes"
print(hdr); print('-' * len(hdr))
for eps in (1.0, 2.0, 4.0, 8.0, 12.0, 16.0, 20.0):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')          # reported in the notes column instead
        g = recommend(eps_y_nm=eps, code='GP', **reference_beam)
        w = recommend(eps_y_nm=eps, code='WX', **reference_beam)
    note = 'GP count beyond what was run' if g.warnings else ''
    print(f'{eps:10g} {g.D_y:7.1f} | {g.n_y:7d} {g.n_m:12,} | {w.n_y:7d} {w.n_m:10,} | {note}')

eps_y [nm]     D_y |  GP n_y       GP n_m |  WX n_y     WX n_m | notes
----------------------------------------------------------------------
         1    97.4 |     512   14,096,339 |     512  1,400,000 | GP count beyond what was run
         2    68.8 |     512    4,471,323 |     256    450,000 | 
         4    48.5 |     256      707,822 |     256    140,000 | 
         8    34.2 |     256      223,511 |     256     46,000 | 
        12    27.9 |     256      113,696 |     256     23,000 | 
        16    24.1 |     256       70,318 |     256     14,000 | 
        20    21.5 |     256       48,411 |     256     10,000 | 


## Your machine

Edit the numbers below and run the cell. Anything outside what this study covered is printed
underneath the result.

In [3]:
my_beam = dict(
    E_GeV      = 125.0,      # beam energy [GeV]
    N          = 6.24e9,     # particles per bunch
    sigma_z_m  = 100e-6,     # bunch length [m]
    eps_x_nm   = 900.0,      # normalised horizontal emittance [nm]
    beta_x_m   = 12e-3,      # horizontal beta at the IP [m]
    eps_y_nm   = 8.0,        # normalised vertical emittance [nm]
    beta_y_m   = 0.12e-3,    # vertical beta at the IP [m]
)

for code in ('GP', 'WX'):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')          # already listed on the result
        print(recommend(code=code, **my_beam), end='\n\n')

GUINEA-PIG++ [recommended]: (n_x, n_y, n_z) = (512, 256, 64), n_t = 6, n_m = 223,511
    D_y = 34.216, D_x = 0.3226, beta_y*/sigma_z = 1.200, sigma_x* = 210.12 nm, sigma_y* = 1.981 nm
    cut multipliers (c_x, c_y, c_z) = (20, 20, 3.5)

WarpX [recommended]: (n_x, n_y, n_z) = (512, 256, 128), n_t = 128, n_m = 46,000
    D_y = 34.216, D_x = 0.3226, beta_y*/sigma_z = 1.200, sigma_x* = 210.12 nm, sigma_y* = 1.981 nm
    cut multipliers (c_x, c_y, c_z) = (16, 16, 8)



## A different box

Pass `cuts=(c_x, c_y, c_z)` to set the box half-extents yourself. The cell counts scale with
them, so the resolution is the one the study established; a box smaller than the study's is
flagged, because it can crop the collision.

In [4]:
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    print(recommend(code='GP', cuts=(40, 40, 7.0), **my_beam))     # twice the box in each direction
    print()
    print(recommend(code='GP', cuts=(20, 10, 3.5), **my_beam))     # half the vertical extent

GUINEA-PIG++ [recommended]: (n_x, n_y, n_z) = (1024, 512, 128), n_t = 6, n_m = 223,511
    D_y = 34.216, D_x = 0.3226, beta_y*/sigma_z = 1.200, sigma_x* = 210.12 nm, sigma_y* = 1.981 nm
    cut multipliers (c_x, c_y, c_z) = (40.0, 40.0, 7.0)
    ! box extents (c_x, c_y, c_z) = (40.0, 40.0, 7.0) are not the (20, 20, 3.5) this study ran; the cell counts have been scaled with them to hold the resolution.

GUINEA-PIG++ [recommended]: (n_x, n_y, n_z) = (512, 128, 64), n_t = 6, n_m = 223,511
    D_y = 34.216, D_x = 0.3226, beta_y*/sigma_z = 1.200, sigma_x* = 210.12 nm, sigma_y* = 1.981 nm
    cut multipliers (c_x, c_y, c_z) = (20.0, 10.0, 3.5)
    ! box extents (c_x, c_y, c_z) = (20.0, 10.0, 3.5) are not the (20, 20, 3.5) this study ran; the cell counts have been scaled with them to hold the resolution. The box is smaller in y, so the collision may be cropped; this study did not test that.


## The two modes

**`recommended`** (the default) applies the loci exactly as published. They were established on
one machine, over `eps_y = 1-20 nm`, at `beta_y*/sigma_z = 1.2`. Inside that envelope the numbers
are the ones the study ran; outside it the result says so.

**`model`** is for a machine whose `beta_y*/sigma_z` differs from 1.2. The compression `R` depends
on that ratio, so the recommendation is carried across to the new geometry by recomputing `R`
from the envelope model. Everything the study established is kept — the constants, the margins,
the exponents — and only the geometry factor changes. **This mode is untested**: no simulation in
the paper checks it, and its output is labelled untested wherever it appears.

Below, the same beam at four times the reference `beta_y*/sigma_z`, both ways.

In [5]:
other_geometry = dict(my_beam, beta_y_m=4 * my_beam['beta_y_m'])

for mode in ('recommended', 'model'):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        print(recommend(code='WX', mode=mode, **other_geometry), end='\n\n')

WarpX [recommended]: (n_x, n_y, n_z) = (512, 256, 128), n_t = 128, n_m = 4,600
    *** OUTSIDE THE RANGE THE STUDY COVERED ***
    D_y = 16.949, D_x = 0.3196, beta_y*/sigma_z = 4.800, sigma_x* = 210.12 nm, sigma_y* = 3.962 nm
    cut multipliers (c_x, c_y, c_z) = (16, 16, 8)
    ! beta_y*/sigma_z = 4.800 differs from the 1.2 the study ran: the compression R, and with it both normalisations, depends on this ratio as well as on D_y, so these constants are outside the geometry they were established on. Use mode='model' for the envelope-model estimate.
    ! D_y = 16.9 is outside the tested band [21.5, 97.4].

WarpX [model]: (n_x, n_y, n_z) = (512, 512, 128), n_t = 128, n_m = 2,800
    *** EXPERIMENTAL: transported to a machine geometry the study did not run ***
    D_y = 16.949, D_x = 0.3196, beta_y*/sigma_z = 4.800, sigma_x* = 210.12 nm, sigma_y* = 3.962 nm
    cut multipliers (c_x, c_y, c_z) = (16, 16, 8)
    compression R: 3.356 at beta_y*/sigma_z = 1.2 -> 11.040 here
    ! EXPERIMENTA

## Limits

- `D_y` outside 21.5-97.4, or `eps_y` outside 1-20 nm, is outside the tuned range.
- A different `beta_y*/sigma_z` needs `mode='model'`, which is untested.
- A box other than the study's is untested, and a smaller one may crop the collision.
- A macroparticle count larger than the study reached is flagged: those rows are extrapolated.
- A beam that is not flat is refused outright. The laws describe the vertical plane of a flat
  beam; on a tall beam the pinch is horizontal, so a vertical requirement would be the wrong
  answer, and the constants would have to be re-established.

In [6]:
try:
    recommend(eps_y_nm=900.0, code='GP', **dict(reference_beam, eps_x_nm=2.0))
except ValueError as e:
    print('refused:', e)

refused: sigma_y* = 21.012 nm is not smaller than sigma_x* = 9.905 nm. These laws are derived for a flat beam, where the vertical plane is the strongly disrupted one; here the pinch would be horizontal, so the vertical requirement returned would apply to the wrong plane.
